# Exploring Weather Data

In [4]:
import pandas as pd

In [5]:
incident_data = pd.read_csv("../../clean_data/florida_incident_data_w_categories.csv")

In [6]:
is_valid_incident = incident_data["incident_type"] != "Other (None of the above)"

In [7]:
is_valid_incident.value_counts()

incident_type
False    8522
True     1459
Name: count, dtype: int64

In [8]:
incident_data[is_valid_incident]["ID"].nunique()

1459

In [21]:
from get_weather import get_weather_data
from datetime import datetime

def get_weather_for_incident(row):

  date_str = row["EventDate"]
  latitude, longitude = row["Latitude"], row["Longitude"]

  if latitude > 90 or latitude < -90 or pd.isna(latitude):
    return {}

  if pd.isna(longitude):
      return {}

  date_converted = pd.to_datetime(date_str, format="%Y-%m-%d")
  weather_data = get_weather_data(latitude, longitude, date_converted)
  data = weather_data.to_dict(orient="records")[0]
  data["associated_incident_id"] = row["ID"]
  return data


In [22]:
valid_incident_data = incident_data[is_valid_incident]

In [26]:
from tqdm.auto import tqdm

# 1. Register tqdm with pandas
tqdm.pandas(desc="Processing rows")

In [ ]:
results = valid_incident_data.progress_apply(get_weather_for_incident, axis=1, result_type="expand")

Processing rows: 100%|██████████| 1459/1459 [00:04<00:00, 294.29it/s]


In [29]:
results.to_csv("valid_incident_weather_info.csv", index=False)